In [1]:

# RESTAURANT CUISINE CLASSIFICATION P


#  Step 1: Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


#  Step 2: Load Dataset
df = pd.read_csv("restaurant.csv")

print(" Dataset loaded successfully!")
print(df.head())
print(df.info())


# Step 3: Handle Missing Values


# Check missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Fill numeric missing values with mean
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Fill categorical missing values with mode
categorical_cols = df.select_dtypes(exclude=[np.number]).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


# Step 4: Select Features and Target


# Target variable: Cuisines
target = 'Cuisines'

# Features: choose relevant ones (avoid address, name, etc.)
features = [
    'Country Code', 
    'City', 
    'Locality', 
    'Longitude', 
    'Latitude',
    'Average Cost for two',
    'Currency',
    'Has Table booking',
    'Has Online delivery',
    'Is delivering now',
    'Price range',
    'Aggregate rating',
    'Rating color',
    'Rating text',
    'Votes'
]

X = df[features]
y = df[target]

# Encode target variable (Cuisines) with LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(y.value_counts())
#step 5:pre process data
categorical_features = ['City', 'Locality', 'Currency', 'Has Table booking',
                        'Has Online delivery', 'Is delivering now', 'Rating color', 'Rating text']
numeric_features = ['Country Code', 'Longitude', 'Latitude', 'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']

# Define preprocessing: impute + one-hot encode categorical, impute numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)
# Take only the first cuisine if multiple are listed (e.g., "North Indian, Chinese" → "North Indian")
df['Main_Cuisine'] = df['Cuisines'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else 'Unknown')

# Group rare cuisines into "Other" (e.g., cuisines with < 5 occurrences)
cuisine_counts = df['Main_Cuisine'].value_counts()
rare_cuisines = cuisine_counts[cuisine_counts < 5].index
df['Main_Cuisine'] = df['Main_Cuisine'].replace(rare_cuisines, 'Other')

print("Unique cuisines after grouping rare ones:", df['Main_Cuisine'].nunique())
print(df['Main_Cuisine'].value_counts().head(10))

# Encode the cuisine labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['Main_Cuisine'])


# Step 6: Train-Test Split


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

#  Step 7: Train Classification Model


#  Random Forest Classifier
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ))
])

clf.fit(X_train, y_train)
print(" Random Forest model trained successfully!")

y_pred = clf.predict(X_test)

print("\n Classification Metrics:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Optional: Confusion matrix (can be large if many cuisines)
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix shape:", cm.shape)

# ============================================
#  Step 9: Analyze Results
# ============================================

# Top predicted cuisines distribution
predicted_cuisine_labels = label_encoder.inverse_transform(y_pred)
print("\nTop 10 Predicted Cuisine Labels:")
print(pd.Series(predicted_cuisine_labels).value_counts().head(10))

# Actual cuisine distribution
print("\nTop 10 Actual Cuisine Labels:")
print(df['Cuisines'].value_counts().head(10))

print("\n Done! You can now analyze which cuisines were hardest to classify (e.g., by looking at precision/recall in the report).")



 Dataset loaded successfully!
   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, 

C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Cuisines
North Indian                          945
North Indian, Chinese                 511
Fast Food                             354
Chinese                               354
North Indian, Mughlai                 334
                                     ... 
World Cuisine, Patisserie, Cafe         1
Burger, Izgara                          1
Desserts, B�_rek                        1
Restaurant Cafe, Turkish, Desserts      1
Restaurant Cafe, Desserts               1
Name: count, Length: 1825, dtype: int64


Unique cuisines after grouping rare ones: 72
Main_Cuisine
North Indian    3001
Chinese          855
Fast Food        672
Bakery           621
Cafe             617
American         278
South Indian     262
Mithai           246
Street Food      236
Continental      235
Name: count, dtype: int64

Training samples: 7640
Testing samples: 1911
 Random Forest model trained successfully!



 Classification Metrics:
Accuracy: 0.33856619570905283

Detailed classification report:
                precision    recall  f1-score   support

       Afghani       0.00      0.00      0.00         1
      American       0.16      0.23      0.19        56
       Arabian       0.00      0.00      0.00         1
         Asian       0.33      0.07      0.11        15
        Awadhi       0.00      0.00      0.00         1
           BBQ       0.00      0.00      0.00         4
        Bakery       0.27      0.02      0.04       124
      Bar Food       0.00      0.00      0.00         2
       Bengali       0.00      0.00      0.00         4
     Beverages       0.00      0.00      0.00        16
       Biryani       0.00      0.00      0.00        22
     Brazilian       0.50      0.25      0.33         4
     Breakfast       0.00      0.00      0.00         5
       British       0.33      0.50      0.40         2
        Burger       0.00      0.00      0.00        23
          Cafe

C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\KEERTHAN\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri